# Research Agent API - Synchronous Client

A robust Python client for the [Bigdata.com Research Agent API](https://docs.bigdata.com/research-agent) that provides synchronous responses with complete citations, automatic retry handling, and network resilience.

## Features

| Feature | Description |
|---------|-------------|
| **Synchronous Interface** | Simple blocking API - no async/await complexity |
| **Automatic Retries** | Exponential backoff for connection errors, timeouts, and server errors |
| **Stream Timeout Detection** | Detects stalled connections and automatically triggers retries |
| **Conversation Continuity** | Resumes interrupted conversations using `chat_id` with the original message |
| **Bigdata.com Citations** | Structured citations with source info, timestamps, and text chunks |
| **Inline Citations** | Answer text with `[1]`, `[2]` markers linked to numbered references |

## Requirements

- Python 3.7+
- `requests` library
- Bigdata.com API key (set as `BIGDATA_API_KEY` environment variable)

## Table of Contents

1. [Setup](#Setup) - Import and configure the client
2. [Retry Mechanism](#Retry-Mechanism-Configuration) - Configure retry behavior for network resilience
3. [Execute Research Query](#Execute-Research-Query) - Run a research query
4. [View Results](#A.-Answer-with-Inline-Citation-Numbers) - Different ways to access results
5. [Save Results](#Save-Results-to-File) - Export to JSON files


## Setup


In [ ]:
import os
import sys
import json
import logging
from IPython.display import display, Markdown, JSON

# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

# Import the client and setup_logging function
from research_client import ResearchClient, setup_logging

# Configure logging using the built-in helper function
# This ensures logs are flushed immediately (important for debugging network issues)
setup_logging(
    log_file="output/research_client.log",  # Log file path
    level=logging.INFO,                      # Log level
    console=True,                            # Also print to console (set False to disable)
    file_mode="w"                            # "w" to overwrite, "a" to append
)

print("✅ Client imported successfully!")
print("✅ Logging configured with immediate flush → output/research_client.log")


In [ ]:
# Create client (reads BIGDATA_API_KEY from environment)
# os.environ["BIGDATA_API_KEY"] = "your-api-key-here"

client = ResearchClient()
print("✅ Client ready")


## Retry Mechanism Configuration

The `ResearchClient` includes built-in retry logic with exponential backoff to handle transient failures:

### Retryable Errors (automatic retry)
- **Connection errors**: Network unreachable, DNS failures
- **Timeouts**: Connection and read timeouts
- **Stream timeout**: No data received within `stream_timeout` period
- **Server errors**: HTTP 500, 502, 503, 504
- **Rate limiting**: HTTP 429 (Too Many Requests)

### Non-Retryable Errors (raised immediately)
- **Client errors**: HTTP 400, 401, 403, 404
- **Invalid parameters**: ValueError

### Default Configuration

| Parameter | Default | Description |
|-----------|---------|-------------|
| `timeout` | 300 | Connection timeout in seconds |
| `stream_timeout` | 60.0 | Max seconds to wait for data during streaming |
| `max_retries` | 3 | Maximum retry attempts |
| `retry_delay` | 1.0 | Initial delay between retries (seconds) |
| `retry_backoff` | 2.0 | Exponential backoff multiplier |
| `retry_max_delay` | 60.0 | Maximum delay cap (seconds) |

### Custom Configuration Example

In [ ]:
# Example: Custom retry configuration for more resilient connections
# Useful for unstable networks or when expecting intermittent issues

client_with_retry = ResearchClient(
    # Timeout settings
    timeout=300,            # Connection timeout: 5 minutes (default: 300)
    stream_timeout=60.0,    # Stream timeout: 60 seconds (default: 30.0)
                            # Triggers retry if no data received for this duration
    
    # Retry settings
    max_retries=5,          # Retry up to 5 times (default: 3)
    retry_delay=2.0,        # Start with 2 second delay (default: 1.0)
    retry_backoff=2.0,      # Double delay after each retry (default: 2.0)
    retry_max_delay=120.0   # Cap delay at 2 minutes (default: 60.0)
)

print("✅ Client with custom configuration ready")
print(f"\n⏱️  Timeout Settings:")
print(f"   Connection timeout: {client_with_retry.timeout}s")
print(f"   Stream timeout: {client_with_retry.stream_timeout}s")
print(f"\n🔄 Retry Settings:")
print(f"   Max retries: {client_with_retry.max_retries}")
print(f"   Initial delay: {client_with_retry.retry_delay}s")
print(f"   Backoff multiplier: {client_with_retry.retry_backoff}x")
print(f"   Max delay cap: {client_with_retry.retry_max_delay}s")
print("\n📊 Retry delay progression (if all retries fail):")
delay = client_with_retry.retry_delay
for i in range(client_with_retry.max_retries):
    actual_delay = min(delay, client_with_retry.retry_max_delay)
    print(f"   Attempt {i+2}: wait {actual_delay:.1f}s before retry")
    delay *= client_with_retry.retry_backoff

### Retry Behavior

The retry mechanism handles the following scenarios automatically:

| Error Type | HTTP Code | Description | Retryable |
|------------|-----------|-------------|-----------|
| `ConnectionError` | - | Network unreachable, DNS failure | ✅ Yes |
| `Timeout` | - | Connection timed out | ✅ Yes |
| `ReadTimeout` | - | No data within read timeout | ✅ Yes |
| `StreamTimeoutError` | - | No data within `stream_timeout` | ✅ Yes |
| `ChunkedEncodingError` | - | Connection broken during streaming | ✅ Yes |
| `HTTPError` | 408 | Request Timeout | ✅ Yes |
| `HTTPError` | 429 | Too Many Requests (rate limit) | ✅ Yes |
| `HTTPError` | 500 | Internal Server Error | ✅ Yes |
| `HTTPError` | 502 | Bad Gateway | ✅ Yes |
| `HTTPError` | 503 | Service Unavailable | ✅ Yes |
| `HTTPError` | 504 | Gateway Timeout | ✅ Yes |
| `HTTPError` | 400 | Bad Request | ❌ No |
| `HTTPError` | 401 | Unauthorized (invalid API key) | ❌ No |
| `HTTPError` | 403 | Forbidden | ❌ No |
| `HTTPError` | 404 | Not Found | ❌ No |

### Conversation Continuity

When a network interruption occurs mid-stream:
1. The client captures any partial data and the conversation `chat_id`
2. On retry, it sends the original message with the `chat_id` to resume
3. Partial responses are accumulated across retries for a complete answer

**Note**: Client errors (4xx except 408/429) are not retried as they indicate issues with the request itself.

In [ ]:
# To monitor retry attempts, enable console logging for the research_client module
# (The default setup only logs to file; this adds console output)

def enable_retry_console_logging():
    """Enable console logging to see retry attempts in real-time."""
    retry_logger = logging.getLogger("research_client")
    
    # Check if console handler already exists
    if not any(isinstance(h, logging.StreamHandler) for h in retry_logger.handlers):
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.WARNING)  # Only show warnings and errors
        console_handler.setFormatter(logging.Formatter(
            '%(asctime)s - %(levelname)s - %(message)s'
        ))
        retry_logger.addHandler(console_handler)
        print("✅ Console logging enabled for retry warnings")
    else:
        print("ℹ️ Console logging already enabled")

# Uncomment to enable console logging for retries:
enable_retry_console_logging()

print("💡 Tip: Enable console logging to see retry attempts in real-time")
print("   When retries occur, you'll see messages like:")
print('   "Retry attempt 1/3 after 1.0s delay"')
print('   "Retryable error on attempt 1/4: ConnectionError: ..."')

## Execute Research Query


In [ ]:
# Execute research
query_message = """ What are the key risks Google is facing? """
#query_message = """ Generate a comprehensive daily macroeconomic morning briefing report for the US market. """


print(f"🔍 Researching: {query_message}")
print("   This may take few seconds...\n")


# NOTE: Additional parameters can be added to the research function based on the requirements.
result = client_with_retry.research(
    message=query_message,
    research_effort=  "standard" # "lite" OR "standard"
)

print(f"✅ Research complete!")
print(f"   Processing time: {result.processing_time_ms}ms")
print(f"   Citations found: {len(result.citations)}")


---
## A. Answer with Inline Citation Numbers

Display the answer with inline citation markers [1], [2], etc. and a numbered references section:


In [ ]:
# Get answer with inline citation numbers
answer_with_citations = result.get_answer_with_citations()

# Get numbered citations that match the inline numbers
numbered_citations = result.get_numbered_citations()

print(f"📊 Found {len(numbered_citations)} inline citations\n")


In [ ]:
# Display answer with inline citation numbers [1], [2], etc.
display(Markdown("## Answer\n\n" + answer_with_citations))


In [ ]:
# Display numbered references section
display(Markdown("---\n## References\n"))

for citation in numbered_citations:
    num = citation.get('number', '?')
    headline = citation.get('headline', 'N/A')
    
    # Build citation card
    parts = [f"**[{num}]** {headline}"]
    
    # Source info
    source = citation.get('source', {})
    source_name = source.get('name') if source else citation.get('source_name')
    if source_name:
        parts.append(f"📰 **{source_name}**")
    
    # Date
    timestamp = citation.get('timestamp')
    if timestamp:
        parts.append(f"📅 {timestamp[:10]}")
    
    # URL
    url = citation.get('url')
    if url:
        parts.append(f"🔗 [{url[:50]}...]({url})")
    
    # Chunks/excerpts
    chunks = citation.get('chunks', [])
    if chunks:
        parts.append("\n**Excerpts:**")
        for chunk in chunks[:2]:  # Show max 2 excerpts
            text = chunk.get('text', '')
            if text:
                display_text = text[:300] + "..." if len(text) > 300 else text
                display_text = display_text.replace('\n', ' ')
                parts.append(f"- *{display_text}*")
    
    display(Markdown("\n".join(parts) + "\n\n---"))


### JSON Export with Inline Citations


In [ ]:
# Export as JSON with inline citations in the answer
result_with_inline = result.to_dict_with_inline_citations()

print(json.dumps(result_with_inline, indent=2)[:3000] + "\n... (truncated)")


In [ ]:
# Save result with inline citations
with open("output/result_with_inline_citations.json", "w") as f:
    f.write(result.to_json_with_inline_citations())
print("✅ Saved: output/result_with_inline_citations.json")


---
## B. Just Response

Display only the research answer (Markdown rendered):


In [ ]:
# Get just the answer
answer = result.get_answer()

display(Markdown(answer))


---
## C. Just Citations

Display only the citations in Bigdata.com format (JSON):


In [ ]:
# Get just the citations as JSON
citations = result.get_citations()

#print first 5 citations   
print(f"📚 Citations ({len(citations)} sources):\n")
print(json.dumps(citations[:5], indent=2))


---
## D. Response with Citations

Display both answer and citations together:


In [ ]:
# Get full result as JSON (answer + citations)
full_result = result.to_dict()

print(json.dumps(full_result, indent=2))


### Formatted View (Answer + Citations)


In [ ]:
# Display answer as Markdown
display(Markdown("## Answer\n" + result.answer))

# Display citations in a readable format
display(Markdown("---\n## Citations"))

for i, citation in enumerate(result.citations[:10], 1):  # Show first 10
    c = citation.to_dict()
    
    # Build citation display
    parts = [f"### [{i}] {c.get('headline', 'N/A')}"]
    
    if c.get('source'):
        src = c['source']
        source_parts = []
        if src.get('name'):
            source_parts.append(f"**Source:** {src['name']}")
        if src.get('rank'):
            source_parts.append(f"**Rank:** {src['rank']}")
        if source_parts:
            parts.append(" | ".join(source_parts))
    
    if c.get('timestamp'):
        parts.append(f"**Date:** {c['timestamp']}")
    
    if c.get('url'):
        parts.append(f"**URL:** {c['url']}")
    
    # Show chunks
    if c.get('chunks'):
        parts.append("\n**Excerpts:**")
        for chunk in c['chunks']:
            text = chunk.get('text', '')
            if text:
                # Truncate long text
                display_text = text[:400] + "..." if len(text) > 400 else text
                display_text = display_text.replace('\n', ' ')
                parts.append(f"- *{display_text}*")
    
    display(Markdown("\n".join(parts) + "\n\n---"))

if len(result.citations) > 10:
    print(f"\n... and {len(result.citations) - 10} more citations")


---
## Save Results to File


In [ ]:
# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

# Save just citations
with open("output/citations.json", "w") as f:
    f.write(result.get_citations_json())
print("✅ Saved: output/citations.json")

# Save full result (answer + citations)
with open("output/research_result.json", "w") as f:
    f.write(result.to_json())
print("✅ Saved: output/research_result.json")


---
## Citation Format Reference

The citations follow the standard Bigdata.com format:

```json
{
  "id": "E91DED180158906A74444B7837742178",
  "headline": "Article Title",
  "timestamp": "2026-01-06T15:00:30",
  "source": {
    "id": "5A5702",
    "name": "Benzinga",
    "rank": "RANK_1"
  },
  "url": "https://...",
  "chunks": [
    {
      "cnum": 5,
      "text": "Relevant text excerpt...",
      "relevance": 0.94,
      "sentiment": 0.82
    }
  ]
}
```

**Fields** (only non-null values are included):
- `id`: Document identifier
- `headline`: Article title
- `timestamp`: Publication date/time
- `source.id`: Source identifier
- `source.name`: Source name (e.g., "Benzinga", "Yahoo! Finance")
- `source.rank`: Source quality rank (e.g., "RANK_1")
- `url`: Document URL
- `chunks`: Array of relevant text excerpts with relevance scores
